# Getting Started: Bond Scan for H2

This notebook performs a simple **bond-length scan** for the hydrogen molecule
**H2** using **VQE**.

Goals:

- define a range of H-H distances
- run VQE at each geometry
- build an energy curve
- identify the minimum-energy bond length in the scanned range

This is one of the most standard small quantum chemistry workflows.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from vqe.core import run_vqe

## Why a bond scan?

A molecule does not have a single arbitrary geometry. Its energy depends on the
nuclear coordinates.

For a diatomic molecule like `H2`, the simplest structural study is to vary the
bond length and compute the energy at each point:

$$
E = E(R)
$$

The minimum of this curve gives the lowest-energy bond length within the chosen
model and basis.

## Distance grid

We choose a small range of H-H distances in angstrom.

In [ ]:
distances = np.linspace(0.40, 1.60, 13)
distances

## VQE settings

We keep the algorithm settings fixed across the scan so that the geometry is
the only changing variable.

In [ ]:
ansatz_name = "UCCSD"
optimizer_name = "Adam"
steps = 60
stepsize = 0.2
seed = 0
basis = "sto-3g"
charge = 0
unit = "angstrom"

## Run the scan

For each bond length `R`, we construct an explicit geometry:

- first H at `(0, 0, 0)`
- second H at `(R, 0, 0)`

Then we run `VQE` and store the final energy.

In [ ]:
energies = []
results = []

for R in distances:
    symbols = ["H", "H"]
    coordinates = np.array(
        [
            [0.0, 0.0, 0.0],
            [R, 0.0, 0.0],
        ],
        dtype=float,
    )

    res = run_vqe(
        molecule=None,
        symbols=symbols,
        coordinates=coordinates,
        basis=basis,
        charge=charge,
        unit=unit,
        ansatz_name=ansatz_name,
        optimizer_name=optimizer_name,
        steps=steps,
        stepsize=stepsize,
        seed=seed,
        noisy=False,
        force=True,
        plot=False,
    )

    energy = float(res["energy"])

    energies.append(energy)
    results.append(
        {
            "distance": float(R),
            "energy": energy,
            "result": res,
        }
    )

    print(f"R = {R:.2f} {unit:8s} -> E = {energy:.10f} Ha")

In [ ]:
energies = np.asarray(energies, dtype=float)
energies

## Minimum on the scanned grid

In [ ]:
imin = int(np.argmin(energies))
best_distance = float(distances[imin])
best_energy = float(energies[imin])

print(f"Lowest scanned energy : {best_energy:.10f} Ha")
print(f"Best scanned distance : {best_distance:.6f} {unit}")

## Energy curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(distances, energies, marker="o")
plt.axvline(best_distance, linestyle="--", label=f"Minimum at {best_distance:.3f} {unit}")
plt.xlabel(f"H-H distance [{unit}]")
plt.ylabel("Energy [Ha]")
plt.title("VQE bond scan for H2")
plt.grid(True)
plt.legend()
plt.show()

## Inspect the lowest-energy geometry

In [ ]:
best_geometry = np.array(
    [
        [0.0, 0.0, 0.0],
        [best_distance, 0.0, 0.0],
    ],
    dtype=float,
)

print("Best geometry:")
print(best_geometry)

## Compare optimization traces for a few representative distances

It is often useful to inspect not only the final energies, but also how the
optimizer behaved at different geometries.

In [ ]:
sample_distances = [0.40, 0.80, 1.20, 1.60]
sample_indices = [int(np.argmin(np.abs(distances - d))) for d in sample_distances]

plt.figure(figsize=(8, 4))
for idx in sample_indices:
    R = float(results[idx]["distance"])
    trace = np.asarray(results[idx]["result"]["energies"], dtype=float)
    plt.plot(np.arange(len(trace)), trace, marker="o", label=f"R = {R:.2f} {unit}")

plt.xlabel("Iteration")
plt.ylabel("Energy [Ha]")
plt.title("VQE convergence traces across the H2 bond scan")
plt.grid(True)
plt.legend()
plt.show()

## Tabulated scan results

In [ ]:
print(f"{'Distance [' + unit + ']':<18} {'Final energy [Ha]':>20}")
print("-" * 40)
for R, E in zip(distances, energies):
    print(f"{R:<18.6f} {E:>20.10f}")

## Interpretation

The energy is high at very short bond lengths because the nuclei are too close.
As the bond is stretched from that regime, the energy decreases toward a
minimum. If the bond is stretched too far, the energy rises again.

This is the basic shape of a molecular potential-energy curve.

## What this notebook showed

We:

- defined a grid of H-H distances
- ran `VQE` at each geometry
- built an energy curve
- identified the lowest-energy bond length on the scan
- compared convergence traces at several distances

This is the basic bond-scan workflow in the repository.

## Next steps

Good follow-ups are:

- use a denser distance grid near the minimum
- compare different ansätze on the same scan
- repeat the scan with `VarQITE`
- build angle scans for larger molecules such as `H2O`